# RDEB skin cells in the five-species SATURN cell holdout

A fresh shared foundation, supervised baseline and InfoNCE compare atlas mouse cells with both WT and knockout study animals. Pretraining provides a reference representation. This notebook reads completed artifacts; training is controlled separately by the six-hour cloud runner.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display, Image, HTML
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/run_cell_holdout_jax.py').is_file())
sys.path.insert(0, str(repo_root))
os.environ.setdefault('IN_DATA', 'data/manifests/hmmff_rdeb_skin_cell_holdout.csv')
os.environ.setdefault('RUN_ROOT', 'out/hmmff_rdeb_skin_cell_holdout')
os.environ.setdefault('SATURN_TRIALS', 'baseline,infonce')
from jax_saturn.experiments.cell_holdout import HoldoutConfig
from disease_pilot.report import verify_disease_report
config = HoldoutConfig.from_env()
root = config.root
assert config.trials == ('baseline', 'infonce')
assert json.loads((root / 'experiment_config.json').read_text()) == config.scientific_record()
metadata_path = repo_root / os.environ.get('DISEASE_METADATA', 'data/disease_pilot/rdeb_skin/cell_holdout_seed0/evaluation_metadata.csv.gz')
disease = verify_disease_report(config, metadata_path)
display(pd.Series(config.scientific_record(), name='experiment'))
display(pd.DataFrame(disease['groups']))

## Input scale, provenance and split

All original mouse atlas cells are retained and joined to study WT and KO cells on the shared measured gene set. Derived copies of all five species sum to 10,000 expression units per cell over retained genes. Study raw counts are retained separately; the inherited count likelihood is a compatibility surrogate for normalized expression.

Ten percent per species is excluded before HVGs, centroids, model fitting, graphs and checkpoint selection. This is a cell holdout: an animal can occur in both splits. The original atlas has no usable donor/assay annotations. Disease metadata never enters InfoNCE training or selection.

In [ ]:
split = pd.read_csv(root / 'split/split.csv', dtype=str, keep_default_na=False)
assert split.obs_id.is_unique
assert set(split['split']) == {'train', 'test'}
display(split.groupby(['species', 'split']).size().unstack())
display(pd.read_csv(root / 'split/disease_composition.csv'))
display(json.loads((root / 'input_record.json').read_text()))
display(json.loads((root / 'preflight.json').read_text()))

## Training and checkpoint selection

Both methods start from the same unlabelled foundation. InfoNCE receives exactly embeddings, macrogenes, species and observation IDs. Selection follows the established training-only mixing/preservation rule. Cell-type labels enter the supervised baseline and post-hoc metrics; healthy and affected cells of the same type share a label.

In [ ]:
from jax_saturn.experiments.hmmff import trial_complete
pretraining = pd.read_csv(root / 'foundation/baseline/saturn_results/pretrain_losses.csv')
assert pretraining.epoch.tolist() == list(range(1, config.pretrain_epochs + 1))
display(pretraining)
for trial in config.trials:
    history = pd.read_csv(root / 'run' / trial / 'history.csv')
    assert history.epoch.tolist() == list(range(1, config.metric_epochs + 1))
    if trial != 'baseline':
        assert trial_complete(root / 'run' / trial, trial, epochs=config.metric_epochs)
    display(history)
display(pd.read_csv(root / 'run/comparison.csv'))
display(json.loads((root / 'run/acceptance.json').read_text()))

## Held-out integration and cell-type performance

Held-out cells query the frozen training reference. Training queries exclude themselves. UMAP fits only training observations, then transforms the held-out population. Absolute scores and species/cell-type breakdowns matter alongside train/test gaps.

In [ ]:
display(pd.read_csv(root / 'heldout_metrics.csv'))
display(pd.read_csv(root / 'diagnostics/generalization.csv').query("dimension == 'overall'"))
for name in ('generalization_summary', 'generalization_by_species', 'generalization_by_cell_type'):
    path = root / 'diagnostics' / f'{name}.png'
    if path.is_file():
        display(Image(filename=str(path)))
display(Image(filename=str(root / 'report_assets/heldout_umaps.png')))

## Mouse condition, source, donor, cell type, tissue and QC

Each column uses a separate representation's training-fitted coordinates; all colourings within that column reuse the same coordinates. Atlas-reference condition is distinct from same-study WT controls. Raw library size and mitochondrial fraction are unavailable for original atlas cells and are shown as missing.

In [ ]:
display(Image(filename=str(root / 'report_assets/disease_overview.png')))

## Primary comparison: keratinocytes within skin

Compare original atlas skin keratinocytes, study WT and study KO. Look for WT alignment and a KO shift present in both KO animals. WT and KO moving together away from the atlas suggests a source/assay/site effect. Inspect donor and QC colourings before attributing a pattern to disease. The skin context view retains the atlas's basal, epidermal and stem annotations as separate labels.

In [ ]:
display(Image(filename=str(root / 'report_assets/skin_context.png')))
display(Image(filename=str(root / 'report_assets/skin_keratinocytes.png')))

## Within-study fibroblasts

The original mouse skin atlas subset has no fibroblast reference. Interpret this view as a study WT/KO and donor comparison; separation from atlas cells alone does not demonstrate disease-state discovery.

In [ ]:
display(Image(filename=str(root / 'report_assets/study_fibroblasts.png')))

## Interactive disease explorer

Filter representation, cell subset, colour, split and donor. Hover to inspect observation identity, source, condition and cell type.

In [ ]:
display(HTML((root / 'report_assets/disease_explorer.html').read_text()))

## Interpretation and reusable artifacts

A disease shift is an exploratory observation, not a required success condition. No distinct disease cluster is a valid result. With two WT and two KO animals, this cell holdout does not establish transfer to unseen animals or a general disease effect. Atlas donor/assay information is unavailable; normalized expression and the inherited likelihood remain limitations.

The run preserves input/split hashes, preprocessing gene order, native checkpoints, frozen embeddings, UMAP reducers and coordinates, group-level diagnostics, PNG/PDF figures, HTML explorers and the executed notebook. Regenerate the report with the same configuration using `python scripts/run_cell_holdout_jax.py report`, then execute this notebook with `scripts/execute_saturn_notebook.py`. Complete checksum-verified backup/download and cloud STOP/cleanup before declaring execution complete.

In [ ]:
display(disease)
display(json.loads((root / 'timing_gate.json').read_text()))